If you want to type along with me, head to [this notebook](https://humboldt.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fmlekimchi%2Fdata111_fall26&branch=main&urlpath=tree%2Fdata111_fall26%2Flectures%2Flec18_live.ipynb) instead. If you prefer follow along by executing the cells, stay in this notebook.

# Lecture 18: Decisions — Hypothesis Testing
v.ekc

Everything we've been doing informally gets its official name today: the **hypothesis test**. Null, alternative, test statistic, p-value — the vocabulary of statistical decisions. (Slides: KL Lecture 18 deck.)

**Today's sections:**
1. Alameda County, concluded
2. The hypothesis-testing framework
3. The instructor's defense
4. Making the decision

### 📋 Board Reference — the hypothesis test (KL deck, slides 10–15)

| Step | Meaning |
|---|---|
| **Null hypothesis** | "nothing special — it's chance"; precise enough to simulate |
| **Alternative** | "something other than chance is going on" |
| **Test statistic** | one number; choose so *big* (or *small*) favors the alternative |
| **Prediction under the null** | simulate the statistic assuming the null |
| **Conclusion** | observed value consistent with simulation → null stands; way out in the tail → reject |

In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

---
## 1. Alameda County, Concluded

The TVD test from last time, start to finish — this *is* a hypothesis test; we just hadn't named the parts yet.

In [ ]:
jury = Table().with_columns(
    'Ethnicity', make_array('Asian', 'Black', 'Latino', 'White', 'Other'),
    'Eligible', make_array(0.15, 0.18, 0.12, 0.54, 0.01),
    'Panels', make_array(0.26, 0.08, 0.08, 0.54, 0.04)
)

jury

In [ ]:
jury.barh('Ethnicity')

In [ ]:
# Under the model, this is the true distribution of people
# from which the jurors are randomly sampled
model = jury.column('Eligible')
model

In [ ]:
# Let's simulate a random draw of 1423 jurors from this distribution
simulated = sample_proportions(1423, model)
simulated

In [ ]:
# The actual observed distribution (Panels) looks quite different
# from the simulation -- try running this several times to confirm!
jury_with_simulated = jury.with_column('Simulated', simulated)
jury_with_simulated

In [ ]:
jury_with_simulated.barh('Ethnicity')

## Distance Between Distributions

In [ ]:
#  In the Mendel model from before, the difference between observed white/purple
# and their expected values (25%/75%) was our statistic.
#
# In this case, we need to understand how each of the 5 categories
# differ from their expected values according to the model.

diffs = jury.column('Panels') - jury.column('Eligible')
jury_with_difference = jury.with_column('Difference', diffs)
jury_with_difference

## Total Variation Distance

In [ ]:
def tvd(dist1, dist2):
    return sum(abs(dist1 - dist2))/2

In [ ]:
# The TVD of our observed data (Panels) from their expected values
# assuming the model is true (Eligbible)
obsvd_tvd = tvd(jury.column('Panels'), jury.column('Eligible'))
obsvd_tvd

In [ ]:
# The TVD of a model simluation from its expected values
tvd(sample_proportions(1423, model), model)

In [ ]:
def simulated_tvd():
    return tvd(sample_proportions(1423, model), model)

tvds = make_array()

num_simulations = 10000
for i in np.arange(num_simulations):
    new_tvd = simulated_tvd()
    tvds = np.append(tvds, new_tvd)

In [ ]:
title = 'Simulated TVDs (if model is true)'
bins = np.arange(0, .05, .005)

Table().with_column(title, tvds).hist(bins = bins)
print('Observed TVD: ' + str(obsvd_tvd))
plots.scatter(0.14,1.3,color='r',s=30)

---
## 2. The Instructor's Defense 🎓

A class of 359 students in 12 sections. Section 3's average midterm score (13.67) is noticeably below the class average (15.49). The TA says: *"it's just chance — my section is like a random sample of the class."* That's a **null hypothesis**, and it's simulatable.

> **Null:** Section 3's average is like the average of 27 students drawn at random from the class.
> **Alternative:** No — it's systematically lower.
> **Test statistic:** the sample average (small values favor the alternative).

In [ ]:
scores = Table.read_table('scores_by_section.csv')
scores

In [ ]:
scores.group('Section')

In [ ]:
scores.group('Section', np.average).show()

In [ ]:
observed_average = 13.6667 

In [ ]:
random_sample = scores.sample(27, with_replacement=False)
random_sample

In [ ]:
np.average(random_sample.column('Midterm'))

In [ ]:
# Simulate one value of the test statistic 
# under the hypothesis that the section is like a random sample from the class
def random_sample_midterm_avg():
    random_sample = scores.sample(27, with_replacement = False)
    return np.average(random_sample.column('Midterm'))

In [ ]:
# Simulate 50,000 copies of the test statistic

sample_averages = make_array()

for i in np.arange(50000):
    sample_averages = np.append(sample_averages, random_sample_midterm_avg())    

---
## 3. Making the Decision

Where does 13.67 fall among 50,000 simulated section averages? Two equivalent ways to decide:

In [ ]:
# Compare the simulated distribution of the statistic
# and the actual observed statistic
averages_tbl = Table().with_column('Random Sample Average', sample_averages)
averages_tbl.hist(bins = 20)
plots.scatter(observed_average, -0.01, color='red', s=120);

### Approach 1 — the p-value

What fraction of simulations were *as extreme or more* than the observed value?

In [ ]:
# (1) Calculate the p-value: simulation area beyond observed value
np.count_nonzero(sample_averages <= observed_average) / 50000
# (2) See if this is less than 5%

### Approach 2 — the 5% cutoff

Find the value that cuts off the bottom 5% of simulations; is the observation below it?

In [ ]:
# (1) Find simulated value corresponding to 5% of 50,000 = 2500
five_percent_point = averages_tbl.sort(0).column(0).item(2500)
five_percent_point

In [ ]:
# (2) See if this value is greater than observed value
observed_average

### Visual Representation

In [ ]:
averages_tbl.hist(bins = 20)
plots.plot([five_percent_point, five_percent_point], [0, 0.35], color='gold', lw=2)
plots.title('Area to the left of the gold line: 5%');
plots.scatter(observed_average, -0.01, color='red', s=120);

> **Conventions** (KL deck, slides 25–28): p < 0.05 → "statistically significant"; p < 0.01 → "highly significant." These cutoffs are customs, not laws of nature — the p-value itself is the honest summary. Here p ≈ 0.05–0.06: right on the boundary, so reasonable people can disagree — and that's a legitimate outcome of a test.

### Try It 1: Name the parts 🏷️

1. For the *Swain v. Alabama* analysis, state the null hypothesis, the alternative, and the test statistic. (No code — just write it in the cell below.)

*1. Your answer: (double-click to edit)*

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Null: panels are drawn at random from the eligible population (26% Black men). Alternative: the process under-selects Black men. Test statistic: number of Black men on a panel of 100 — small values favor the alternative.*

</details>

---
> **Today's takeaway:** null + alternative + test statistic + simulation + comparison = hypothesis test. You've been doing it for a week; now you can name every part. Homework 7 asks you to run this recipe solo.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Minimal template — p-value
```python
p_value = np.count_nonzero(simulated_stats <= observed_stat) / len(simulated_stats)
```